In [1]:
#importing required python libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import struct
import glob
import json
import pyubx2

In [2]:
df_m10_decoded = pd.read_json("/Users/eagmurray/Projects/DataAnalysis/Q40/flight_tests_12_May/noAirRaid/Flight_2_noAR_30mAlt/decoded_log.jsonl",lines=True)
df_m10_raw = pd.read_json("/Users/eagmurray/Projects/DataAnalysis/Q40/flight_tests_12_May/noAirRaid/Flight_2_noAR_30mAlt/raw_log.jsonl",lines=True)

In [3]:
import pandas as pd
import struct

INPUT_FILE = "ubx_raw.jsonl"
OUTPUT_FILE = "ubx_raw_sanitised.jsonl"

def ubx_checksum(data: bytes):
    ck_a = 0
    ck_b = 0
    for b in data:
        ck_a = (ck_a + b) & 0xFF
        ck_b = (ck_b + ck_a) & 0xFF
    return ck_a, ck_b

def sanitise_nav_pvt(hex_string: str) -> str:
    frame = bytearray(bytes.fromhex(hex_string))

    # Basic UBX check
    if frame[0] != 0xB5 or frame[1] != 0x62:
        return hex_string  # skip non-UBX safely

    msg_class = frame[2]
    msg_id = frame[3]

    # Only touch NAV-PVT
    if (msg_class, msg_id) != (0x01, 0x07):
        return hex_string

    payload_len = struct.unpack("<H", frame[4:6])[0]
    payload_start = 6

    # Offsets inside payload
    lon_offset = payload_start + 24
    lat_offset = payload_start + 28

    # Zero coordinates
    frame[lon_offset:lon_offset+4] = struct.pack("<i", 0)
    frame[lat_offset:lat_offset+4] = struct.pack("<i", 0)

    # Recompute checksum
    ck_a, ck_b = ubx_checksum(frame[2:-2])
    frame[-2] = ck_a
    frame[-1] = ck_b

    return frame.hex()

In [4]:
# Load
# df = pd.read_json("M10_1718March/raw_log.jsonl", lines=True)

# Apply
df_out = df_m10_raw.copy()
df_out["UBX"] = df_out["UBX"].apply(sanitise_nav_pvt)

# Save
df_out.to_json(OUTPUT_FILE, orient="records", lines=True)

print("Done — NAV-PVT coordinates sanitised.")

Done — NAV-PVT coordinates sanitised.


In [5]:
sample = bytes.fromhex(df_out["UBX"].iloc[0])

lon = struct.unpack("<i", sample[6+24:6+28])[0]
lat = struct.unpack("<i", sample[6+28:6+32])[0]

print("lon:", lon, "lat:", lat)

lon: 0 lat: 0
